# 02 · Optimización de hiperparámetros con Hyperband

**Tema 2 del proyecto.** Se buscan los mejores hiperparámetros con **Hyperband** (KerasTuner):

| Hiperparámetro | Espacio de búsqueda |
|---|---|
| Modelo | 3 CNN (propia, EfficientNetB0, ResNet50V2) y 2 transformers (ViT-B/16, DeiT-S) |
| Tasa de aprendizaje | 1e-5 a 3e-3 (escala logarítmica) |
| Tamaño de lote | 16, 32 (64 no cabe en memoria con ViT-B/16) |
| Dropout | 0.0 a 0.5 |
| Épocas | las asigna Hyperband (1, 3 o 9 según la ronda) |

La búsqueda usa **solo** el subconjunto de train (10.000 imágenes) y el de validación (2.500).
Test no interviene.

**Requisitos de la sesión:** Accelerator = GPU T4 x2, Internet = On (descarga de pesos
preentrenados) y, como Input, el dataset y la salida del notebook `01-preparacion-datos`.

**Salida:** `rice_models.py`, `hyperband_trials.csv` y `best_hparams.csv`.

In [ ]:
import os

# Asignador de memoria que devuelve la GPU al soltar cada modelo (evita fragmentación entre trials).
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("keras_tuner") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "keras-tuner"], check=True)

import gc
import time
from pathlib import Path

import keras
import keras_tuner as kt
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
print("Keras", keras.__version__, "| TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Cargar la partición y el generador del notebook 01

In [ ]:
def find_splits_dir(base="/kaggle/input"):
    for root, _, files in os.walk(base):
        if "subset_train.csv" in files and "rice_data.py" in files:
            return Path(root)
    raise FileNotFoundError("Agrega como Input la salida del notebook 01-preparacion-datos")

SPLITS = find_splits_dir()
sys.path.append(str(SPLITS))
from rice_data import CLASSES, IMG_SIZE, SEED, find_data_root, make_dataset

DATA_ROOT = find_data_root()
sub_train = pd.read_csv(SPLITS / "subset_train.csv")
sub_val = pd.read_csv(SPLITS / "subset_val.csv")
print("Particiones:", SPLITS)
print("Dataset:", DATA_ROOT)
print("Subconjunto train:", len(sub_train), "| subconjunto val:", len(sub_val))

## 2. Módulo compartido `rice_models.py`

Define los 5 modelos en un solo lugar para que los notebooks de validación cruzada,
entrenamiento final y ensambles construyan exactamente las mismas arquitecturas.

- **cnn_propia:** CNN de 4 bloques entrenada desde cero.
- **efficientnetb0, resnet50v2:** CNN preentrenadas en ImageNet (`keras.applications`).
- **vit_b16, deit_s:** transformers preentrenados en ImageNet (`keras_hub`).

Todos reciben píxeles en [0, 255] y aplican por dentro su propia normalización.

In [ ]:
%%writefile rice_models.py
"""Los 5 modelos del proyecto Rice Image Dataset (3 CNN y 2 transformers)."""
import keras
import keras_hub
from keras import layers

NUM_CLASSES = 5
IMG_SIZE = 224
MODEL_NAMES = ["cnn_propia", "efficientnetb0", "resnet50v2", "vit_b16", "deit_s"]
HUB_PRESETS = {
    "vit_b16": "vit_base_patch16_224_imagenet",
    "deit_s": "deit_small_distilled_patch16_224_imagenet",
}


def _head(x, dropout):
    x = layers.Dropout(dropout)(x)
    return layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)


def _cnn_propia(dropout):
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1 / 255)(inputs)
    for filters in (32, 64, 128, 256):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, _head(x, dropout), name="cnn_propia")


def _efficientnetb0(dropout):
    # EfficientNet de keras.applications ya normaliza por dentro.
    base = keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    return keras.Model(inputs, _head(base(inputs), dropout), name="efficientnetb0")


def _resnet50v2(dropout):
    base = keras.applications.ResNet50V2(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1 / 127.5, offset=-1)(inputs)
    return keras.Model(inputs, _head(base(x), dropout), name="resnet50v2")


def _hub(name, dropout):
    return keras_hub.models.ImageClassifier.from_preset(
        HUB_PRESETS[name], num_classes=NUM_CLASSES, activation="softmax",
        dropout=dropout, head_dtype="float32")


def build_model(name, lr=1e-4, dropout=0.2):
    """Construye y compila uno de los modelos de MODEL_NAMES."""
    if name == "cnn_propia":
        model = _cnn_propia(dropout)
    elif name == "efficientnetb0":
        model = _efficientnetb0(dropout)
    elif name == "resnet50v2":
        model = _resnet50v2(dropout)
    elif name in HUB_PRESETS:
        model = _hub(name, dropout)
    else:
        raise ValueError(f"Modelo desconocido: {name}")
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
from rice_models import MODEL_NAMES, build_model

# Precisión mixta: cálculos en float16 en la GPU, cerca del doble de velocidad en una T4.
keras.mixed_precision.set_global_policy("mixed_float16")
keras.utils.set_random_seed(SEED)

## 3. Prueba rápida de los 5 modelos

Antes de gastar cuota en la búsqueda, cada modelo se construye y entrena con 2 lotes.
Si alguno falla (pesos que no descargan, memoria insuficiente), el error aparece aquí.

In [ ]:
prueba = []
for name in MODEL_NAMES:
    keras.backend.clear_session()
    gc.collect()
    t0 = time.time()
    model = build_model(name)
    # Lote de 32: el más grande del espacio de búsqueda, así se prueba el peor caso de memoria.
    ds = make_dataset(sub_train.head(128), DATA_ROOT, batch_size=32, training=True).take(2)
    hist = model.fit(ds, epochs=1, verbose=0)
    prueba.append({"modelo": name, "parámetros (M)": round(model.count_params() / 1e6, 2),
                   "loss": round(hist.history["loss"][0], 3), "segundos": round(time.time() - t0)})
    print("OK", name)
    del model
pd.DataFrame(prueba)

## 4. Búsqueda con Hyperband

Hyperband reparte el presupuesto así: arranca muchas configuraciones con pocas épocas,
conserva la tercera parte con mejor `val_loss` (`factor=3`) y les da más épocas, hasta llegar a
`max_epochs`. Así descarta pronto las malas configuraciones sin entrenarlas completas.

Se optimiza `val_loss` y no la exactitud porque, en este dataset, casi todas las
configuraciones superan 99 % y la exactitud no alcanza a distinguirlas.

In [ ]:
class RiceHyperModel(kt.HyperModel):
    def build(self, hp):
        keras.backend.clear_session()
        gc.collect()
        return build_model(
            hp.Choice("model", MODEL_NAMES),
            lr=hp.Float("lr", 1e-5, 3e-3, sampling="log"),
            dropout=hp.Float("dropout", 0.0, 0.5, step=0.1),
        )

    def fit(self, hp, model, *args, **kwargs):
        batch_size = hp.Choice("batch_size", [16, 32])
        return model.fit(
            make_dataset(sub_train, DATA_ROOT, batch_size=batch_size, training=True),
            validation_data=make_dataset(sub_val, DATA_ROOT, batch_size=64),
            *args, **kwargs)


tuner = kt.Hyperband(
    RiceHyperModel(),
    objective="val_loss",
    max_epochs=9,
    factor=3,
    hyperband_iterations=1,
    seed=SEED,
    directory="/tmp/hyperband",  # fuera de /kaggle/working: los checkpoints pesan varios GB
    project_name="rice",
)
tuner.search_space_summary()

In [ ]:
t0 = time.time()
tuner.search(
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=2)],
    verbose=2,
)
print(f"Búsqueda terminada en {(time.time() - t0) / 60:.0f} min")

## 5. Resultados

In [ ]:
filas = []
for trial in tuner.oracle.trials.values():
    if trial.score is None:
        continue
    hp = trial.hyperparameters.values
    acc = trial.metrics.get_best_value("val_accuracy") if trial.metrics.exists("val_accuracy") else None
    filas.append({
        "trial": trial.trial_id, "model": hp["model"], "lr": hp["lr"],
        "dropout": hp["dropout"], "batch_size": hp["batch_size"],
        "epochs": hp.get("tuner/epochs"), "val_loss": trial.score, "val_accuracy": acc,
    })

trials = pd.DataFrame(filas).sort_values("val_loss").reset_index(drop=True)
trials.to_csv(OUT_DIR / "hyperband_trials.csv", index=False)
print("Configuraciones evaluadas:", len(trials))
trials.head(10)

Mejor configuración **de cada modelo**. Es la que usan los notebooks siguientes. Si Hyperband
no llegó a probar algún modelo, se le asigna la mediana de las 5 mejores configuraciones y
queda marcado en la columna `origen`.

In [ ]:
best = trials.loc[trials.groupby("model")["val_loss"].idxmin()].copy()
best["origen"] = "hyperband"

top = trials.head(5)
faltantes = [
    {"model": name, "lr": top["lr"].median(), "dropout": top["dropout"].median(),
     "batch_size": int(top["batch_size"].median()), "origen": "mediana top 5"}
    for name in MODEL_NAMES if name not in set(best["model"])
]
if faltantes:
    best = pd.concat([best, pd.DataFrame(faltantes)], ignore_index=True)

best = best.set_index("model").loc[MODEL_NAMES].reset_index()
best.to_csv(OUT_DIR / "best_hparams.csv", index=False)
best[["model", "lr", "dropout", "batch_size", "epochs", "val_loss", "val_accuracy", "origen"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, grupo in trials.groupby("model"):
    axes[0].scatter(grupo["lr"], grupo["val_loss"], label=name, s=45)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Tasa de aprendizaje")
axes[0].set_ylabel("val_loss")
axes[0].set_title("Cada punto es una configuración evaluada")
axes[0].legend()

trials["epochs"].value_counts().sort_index().plot.bar(ax=axes[1], rot=0)
axes[1].set_xlabel("Épocas asignadas por Hyperband")
axes[1].set_ylabel("Configuraciones")
axes[1].set_title("Reparto del presupuesto")
plt.tight_layout()
plt.show()